# Nhiệm vụ 2: Phân tích Tương quan & So sánh Chuyên sâu Cấu hình - Giá bán (Task 2.1 & 2.2)

**Thành viên thực hiện**: TV3 (Exploratory Data Analysis - EDA)  
**Mã học phần**: IT4930 - Khoa học Dữ liệu & Trí tuệ Kinh doanh  
**Tập dữ liệu**: `data/processed/products_processed.jsonl` ($N = 10.316$ listings)  
**Tài liệu quy chuẩn**: [docs/tv3/tv3.md](file:///d:/NMKHDL/it4930-ecommerce-data-intelligence/docs/tv3/tv3.md) & [docs/feature_contracts_tv3_tv8.md](file:///d:/NMKHDL/it4930-ecommerce-data-intelligence/docs/feature_contracts_tv3_tv8.md)

---
### Mục tiêu nghiên cứu:
1. **Tương quan Bivariate (Pearson & Spearman - Task 2.1)**: Đo lường độ mạnh và hướng liên kết giữa các biến số cấu hình (`ram`, `storage`, `screen_size`, `battery`, `charger`) với biến Giá (`price_vnd` và $\log_{10}(\text{price})$).
2. **Phân tầng Phần cứng (Hardware Stratification - Task 2.1)**: Phân tích biến thiên giá theo từng cấp bậc phần cứng (RAM tiers, Storage tiers, CPU families, GPU classes) và Thương hiệu.
3. **Kiểm định Thống kê Nghiêm ngặt (Task 2.1)**: Thực hiện One-Way ANOVA F-test (tham số) và Kruskal-Wallis H-test (phi tham số) để khẳng định sự khác biệt giá giữa các nhóm cấu hình có ý nghĩa thống kê ($p < 0.05$).
4. **So sánh Đối đầu Chuyên sâu (Laptop vs. Smartphone - Task 2.2)**: Định lượng và đối chiếu sự phân hóa cấu trúc định giá: Laptop (Tài nguyên xử lý: CPU, GPU, RAM, Tản nhiệt/Sạc) vs. Smartphone (Trải nghiệm & Thương hiệu: Brand, ROM, Màn hình OLED). Giải thích các nghịch lý phần cứng (Pin mAh, Camera MP).
5. **Tuyên bố Khoa học Bắt buộc**: Các tương quan và phân tầng là **bằng chứng khám phá thực nghiệm (exploratory associations)**, không cấu thành suy diễn nhân quả (causal inference).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Thiết lập đường dẫn import module EDA nội bộ
eda_scripts_path = Path("../scripts").resolve()
if str(eda_scripts_path) not in sys.path:
    sys.path.append(str(eda_scripts_path))

from eda_stats import (
    load_processed_products,
    compute_bivariate_correlations,
    classify_laptop_hardware,
    classify_smartphone_hardware,
    compute_group_price_stats,
    perform_hypothesis_tests,
    compute_head_to_head_comparison,
    build_pricing_structure_matrix
)
from eda_viz import (
    set_plotting_theme,
    plot_hardware_correlation_heatmaps,
    plot_hardware_price_relationships_laptop,
    plot_hardware_price_relationships_smartphone,
    plot_pricing_drivers_comparison
)

set_plotting_theme()
print("[*] Hệ thống thư viện EDA Task 2 (2.1 & 2.2) đã sẵn sàng!")

## 1. Tải và Phân loại Cấu hình Phần cứng
Phân tầng chi tiết các nhóm RAM, Storage, CPU Tier và GPU Type cho Laptop, cũng như Storage, RAM, Top Brand và Công nghệ Màn hình cho Smartphone.

In [ ]:
df = load_processed_products("../../../data/processed/products_processed.jsonl")
laptop_df = classify_laptop_hardware(df)
phone_df = classify_smartphone_hardware(df)

print(f"Tổng số Laptop: {len(laptop_df):,} | Tổng số Smartphone: {len(phone_df):,}")

## 2. Ma trận Tương quan Bivariate (Pearson & Spearman)
Đo lường mức độ tương quan giữa các thông số cấu hình số với biến mục tiêu $\log_{10}(\text{price})$.
- **Laptop**: Trọng số giá dẫn dắt bởi RAM ($\rho = +0.80$), Pin Wh ($\rho = +0.66$), Sạc W ($\rho = +0.72$) và Ổ cứng ($\rho = +0.52$).
- **Smartphone**: Trọng số giá dẫn dắt bởi Bộ nhớ ROM ($\rho = +0.70$) và RAM ($\rho = +0.64$). Ngược lại, Dung lượng Pin mAh hầu như không tương quan ($\rho = +0.04$).

In [ ]:
laptop_num_cols = ["installed_ram_gb", "storage_gb", "battery_wh", "screen_size_inch", "charger_watt"]
phone_num_cols = ["installed_ram_gb", "storage_gb", "battery_mah", "screen_size_inch", "charger_watt", "camera_mp"]

corr_laptop = compute_bivariate_correlations(df, "laptop", laptop_num_cols)
corr_phone = compute_bivariate_correlations(df, "smartphone", phone_num_cols)

display(corr_laptop)
display(corr_phone)

## 3. Biểu đồ Heatmap Ma trận Tương quan Phi tuyến (Spearman Rank)
So sánh trực quan hai ma trận tương quan giữa Laptop và Smartphone.

In [ ]:
plot_hardware_correlation_heatmaps(df, "../outputs/figures/fig_2_1_hardware_correlation_heatmaps.png")
from IPython.display import Image
Image("../outputs/figures/fig_2_1_hardware_correlation_heatmaps.png")

## 4. Phân tích Biến thiên Giá theo Phân tầng Cấu hình Laptop
Khảo sát bước nhảy giá theo RAM, Storage, CPU Tier và GPU Type.

In [ ]:
plot_hardware_price_relationships_laptop(laptop_df, "../outputs/figures/fig_2_1_hardware_price_relationships_laptop.png")
Image("../outputs/figures/fig_2_1_hardware_price_relationships_laptop.png")

## 5. Phân tích Biến thiên Giá theo Phân tầng Cấu hình & Thương hiệu Smartphone
Khảo sát bước nhảy giá theo ROM, RAM, Top 8 Thương hiệu và Công nghệ Màn hình.

In [ ]:
plot_hardware_price_relationships_smartphone(phone_df, "../outputs/figures/fig_2_1_hardware_price_relationships_smartphone.png")
Image("../outputs/figures/fig_2_1_hardware_price_relationships_smartphone.png")

## 6. Kiểm định Giả thuyết Thống kê (ANOVA & Kruskal-Wallis)
Kiểm định xem sự khác biệt giá giữa các phân nhóm cấu hình/thương hiệu có thực sự mang ý nghĩa thống kê hay không ($p < 0.05$).

In [ ]:
df_classified = pd.concat([laptop_df, phone_df], axis=0, ignore_index=True)
test_factors = [
    ("laptop", "Phân tầng CPU (cpu_tier)", "cpu_tier"),
    ("laptop", "Phân loại GPU (gpu_type)", "gpu_type"),
    ("laptop", "Dung lượng RAM (ram_group)", "ram_group"),
    ("laptop", "Dung lượng Ổ cứng (storage_group)", "storage_group"),
    ("laptop", "Thương hiệu Laptop (brand_norm)", "brand_norm"),
    ("smartphone", "Bộ nhớ trong ROM (storage_group)", "storage_group"),
    ("smartphone", "Dung lượng RAM (ram_group)", "ram_group"),
    ("smartphone", "Thương hiệu Smartphone (brand_norm)", "brand_norm"),
    ("smartphone", "Công nghệ Màn hình (display_group)", "display_group"),
]

tests_summary = perform_hypothesis_tests(df_classified, test_factors)
display(tests_summary)

## 7. So sánh Chuyên sâu Cấu trúc Quyết định Giá: Laptop vs. Smartphone (Nhiệm vụ 2.2)
Lập bảng đối chiếu toàn diện và trực quan hóa so sánh cấu trúc định giá giữa hai ngành hàng.

In [ ]:
# Tính toán ma trận đối chiếu tương quan và nhân tố phân hóa giá
corr_comp_df, top_drivers_df = compute_head_to_head_comparison(df)
matrix_df = build_pricing_structure_matrix()

print("=== BẢNG 2.2A: ĐỐI CHIẾU TƯƠNG QUAN PHẦN CỨNG - GIÁ BÁN ===")
display(corr_comp_df)

print("\n=== BẢNG 2.2B: CÁC NHÂN TỐ PHÂN HÓA GIÁ MẠNH NHẤT & ĐÒN BẨY GIÁ ===")
display(top_drivers_df)

print("\n=== BẢNG 2.2C: MA TRẬN ĐỐI CHIẾU CẤU TRÚC ĐỊNH GIÁ TOÀN DIỆN ===")
display(matrix_df)

In [ ]:
# Trực quan hóa đối đầu chuyên sâu cấu trúc quyết định giá (Fig 2.2)
plot_pricing_drivers_comparison(df, "../outputs/figures/fig_2_2_pricing_drivers_comparison.png")
from IPython.display import Image
Image("../outputs/figures/fig_2_2_pricing_drivers_comparison.png")

## 7. So sánh Chuyên sâu Cấu trúc Quyết định Giá: Laptop vs. Smartphone (Nhiệm vụ 2.2)
Lập bảng đối chiếu toàn diện và trực quan hóa so sánh cấu trúc định giá giữa hai ngành hàng.

In [ ]:
# Tính toán ma trận đối chiếu tương quan và nhân tố phân hóa giá
corr_comp_df, top_drivers_df = compute_head_to_head_comparison(df)
matrix_df = build_pricing_structure_matrix()

print("=== BẢNG 2.2A: ĐỐI CHIẾU TƯƠNG QUAN PHẦN CỨNG - GIÁ BÁN ===")
display(corr_comp_df)

print("\n=== BẢNG 2.2B: CÁC NHÂN TỐ PHÂN HÓA GIÁ MẠNH NHẤT & ĐÒN BẨY GIÁ ===")
display(top_drivers_df)

print("\n=== BẢNG 2.2C: MA TRẬN ĐỐI CHIẾU CẤU TRÚC ĐỊNH GIÁ TOÀN DIỆN ===")
display(matrix_df)

In [ ]:
# Trực quan hóa đối đầu chuyên sâu cấu trúc quyết định giá (Fig 2.2)
plot_pricing_drivers_comparison(df, "../outputs/figures/fig_2_2_pricing_drivers_comparison.png")
from IPython.display import Image
Image("../outputs/figures/fig_2_2_pricing_drivers_comparison.png")